# Magyar Wikipédia RAG

A notebook a repó moduljait használja, új logikát nem tartalmaz. Két részből áll:

1. **Egy kérdés végigvitele a láncon** (keresés, majd válaszgenerálás). Futó Qdrant (`docker compose up -d`), betöltött korpusz (`python -m rag.ingest --recreate`) és futó Ollama (`ollama pull gemma3:4b`) kell hozzá.
2. **A mentett eredmények megtekintése.** Ez külső szolgáltatás nélkül is fut.

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

import pandas as pd
pd.set_option("display.width", 160)
pd.set_option("display.max_colwidth", 80)

## 1. Egy kérdés végigvitele a láncon

In [2]:
from qdrant_client import QdrantClient

from rag.config import get_settings
from rag.embedding import Embedder
from rag.retrieval import DenseRetriever, build_retriever, fetch_chunks

settings = get_settings()
client = QdrantClient(url=settings.qdrant_url)
chunks = fetch_chunks(client, settings.collection)
print(f"{len(chunks)} chunk a(z) '{settings.collection}' gyűjteményben")

# A legjobb keresési mód: dense + szótövezett BM25, RRF-fel egyesítve.
retriever = build_retriever(
    "hybrid_stem5",
    chunks,
    lambda: DenseRetriever(client, Embedder(settings.embedding_model), settings.collection),
)

c:\SZE\MI MSC\LLM\llm-rag-project\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


5419 chunk a(z) 'corpus' gyűjteményben


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 4836.29it/s]


In [3]:
question = "Milyen anyagból, kőzetből építették az egri vár kőfalait?"

hits = retriever.search(question, 5)
pd.DataFrame(
    [{"cikk": h.title, "szakasz": h.section, "pontszám": round(h.score, 4), "szöveg": h.text[:90] + "…"} for h in hits]
)

,cikk,szakasz,pontszám,szöveg
0,Egri vár,A vár története,0.0328,A székesegyházról a 14. századi krónikákban a várat is Szent János evangélis...
1,Egri vár,A vár története,0.0311,Az Árpád-kor elején az egri püspök menedékvára a felsőtárkányi Várhegy hatal...
2,Egri vár,A vár története,0.0290,"A tüzérség megjelenésével a klasszikus lovagvár jellegű (magas, vékony falak..."
3,Egri vár (Pilis),,0.0290,"Az egri vár másolata a Pilisben, a Nagy-Kevély délnyugati lejtőjén, Pilisbor..."
4,Egri vár,A vár története,0.0285,"Falait sárba rakott kőből építették. A templom keleti végét kis, patkó alakú..."


In [4]:
from rag.generate import build_plain_messages, build_rag_messages, ollama_chat

MODEL = "gemma3:4b"  # a 12B változat pontosabb, de CPU-n kb. háromszor lassabb

rag = ollama_chat(build_rag_messages(question, hits, prompt="balanced"), model=MODEL)
plain = ollama_chat(build_plain_messages(question), model=MODEL)

print("RAG-gal:        ", rag.text)
print("Kontextus nélkül:", plain.text)

RAG-gal:         A székesegyház kőfalai riolittufa, az egri borvidék fő talajképző anyagából készültek.
Kontextus nélkül: Nem tudom.


## 2. A mentett eredmények

### 2.1 Retrieval-kísérletek
A táblázat a 44 válaszolható kérdésre mért chunkszintű és cikkszintű mutatókat tartalmazza (`experiments/results/summary.csv`).

In [5]:
summary = pd.read_csv(ROOT / "experiments" / "results" / "summary.csv", encoding="utf-8-sig")
summary.sort_values("chunk_MRR", ascending=False).reset_index(drop=True)

,konfiguráció,chunkok,chunk_R@1,chunk_R@3,chunk_R@5,chunk_R@10,chunk_MRR,cikk_R@1,cikk_MRR
0,corpus_c300_o50 / dense,9187,0.826,0.958,0.958,0.989,0.917,0.951,1.000
1,corpus / hybrid_stem5,5419,0.826,0.913,0.955,0.989,0.904,0.951,1.000
2,corpus_c1000_o200 / dense,2756,0.792,0.936,0.958,0.981,0.890,0.917,0.973
3,corpus_c500_o200 / dense,6655,0.792,0.924,0.981,0.989,0.885,0.951,1.000
4,corpus_c800_o150 / dense,3355,0.701,0.970,0.981,0.992,0.856,0.894,0.966
5,corpus / bm25_stem5,5419,0.758,0.890,0.890,0.920,0.855,0.860,0.955
6,corpus / dense (baseline),5419,0.735,0.913,0.955,1.000,0.849,0.928,0.989
7,corpus / hybrid,5419,0.761,0.845,0.886,0.920,0.827,0.875,0.939
8,corpus_c500_o0 / dense,5132,0.655,0.924,0.989,1.000,0.805,0.951,1.000
9,corpus_c500_o100_notitle / dense,5419,0.689,0.735,0.811,0.841,0.758,0.905,0.977


### 2.2 Generálás
A `gemma3:12b` modell kézi értékelése (1 = helyes, 0,5 = részben helyes, 0 = téves vagy elutasított), RAG-gal és kontextus nélkül.

In [6]:
gen = pd.read_csv(
    ROOT / "experiments" / "generation" / "gemma3-12b__corpus__hybrid_stem5__k5__balanced.csv",
    sep=";", encoding="utf-8-sig", decimal=",",
)
gen["kézi"] = pd.to_numeric(gen["kézi"], errors="coerce")
gen["válaszolható"] = ~gen["típus"].str.contains("nincs", case=False)

gen.groupby(["mód", "válaszolható"])["kézi"].agg(["count", "sum", "mean"]).round(2)

count   sum  mean
mód   válaszolható                   
norag False             6   0.0  0.00
      True             44   3.0  0.07
rag   False             6   5.5  0.92
      True             44  41.0  0.93

In [7]:
# Kérdések, ahol a RAG-os válasz nem teljesen helyes
gen[(gen["mód"] == "rag") & (gen["kézi"] < 1)][["id", "típus", "kérdés", "elvárt válasz", "modell válasza", "kézi"]]

,id,típus,kérdés,elvárt válasz,modell válasza,kézi
9,q005,tény,Melyik két híres európai palotához hasonlítják a fertődi Esterházy-kastélyt?,A bécsi Schönbrunnhoz és a párizsi Versailles-hoz.,A megadott forrásokból nem derül ki.,0.0
81,q041,több cikk,Melyik a régebbi: a Bakonycsernye melletti Csiklingvár vagy a tatai vár?,"A Csiklingvár (mai vára 1260 körül épült, a tatai várat 1397–1409 között épí...","A Csiklingvár már az avar időkben állt itt egy földvárként, míg a tatai vára...",0.5
85,q043,több cikk,Melyik kastély építése kezdődött korábban: a gödöllői Grassalkovich-kastélyé...,A gödöllői (1735 után kezdték építeni; a fertődi vadászkastély átépítése 176...,"A gödöllői Grassalkovich-kastély a 18. században épült, míg a fertődi Esterh...",0.0
87,q044,több cikk,Mely várakat zálogosították el a 15. században Rozgonyi Istvánnak?,"A tatai várat (1426, Zsigmond király), a dédesi várat (1431, Borbála királyn...","Rozgonyi Istvánnak a 15. században a Vitányi várat, a Tatai várat, a Solymár...",0.5
89,q045,nincs válasz,Mennyibe kerül a belépőjegy az egri várba?,A korpusz alapján nem megválaszolható.,A Dobó István Vármúzeum közös múzeumi belépőjegyével látogathatóak a kiállít...,0.5
